In [41]:
from typing import TypedDict

from dotenv import load_dotenv
import os

from langgraph.graph import state
from openai import OpenAI
from langgraph.graph import *
env = load_dotenv()
api_key = os.getenv("API_KEY")
openai_client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")


In [42]:
def do_api_call(system_role:str, system_message:str, user_role:str, user_content:str) ->str:
    completion = openai_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": system_role,
                "content": system_message,
            },
            {
                "role": user_role,
                "content": user_content,
            }
        ],
        reasoning_effort=None,
        temperature=0.1
    )
    return completion.choices[0].message.content

In [43]:
# response = do_api_call("developer","Talk like a pirate.", "user", "How do I check if a Python object is an instance of a class?")
# print(response)

In [45]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END


# ---------------- STATE ---------------- #

class GraphState(TypedDict):
    user_role: str
    user_message: str
    output_value: Optional[str]


# ---------------- TOOL ---------------- #

def answering_agent(user_role: str, user_content: str) -> str:

    sys_msg = """
    Answer all user queries.

    If user asks about any model-related query,
    say a sarcastic story.
    """

    response = do_api_call(
        system_role="system",
        system_message=sys_msg,
        user_role=user_role,
        user_content=user_content
    )

    return response


def build_json(user_role: str, user_content: str) -> str:

    sys_msg = """
    You are a JSON builder agent.

    Convert the given response into JSON format.

    Return only final answer.
    Do not expose thinking.
    Do not output <think> tags.
    """

    response = do_api_call(
        system_role="system",
        system_message=sys_msg,
        user_role=user_role,
        user_content=user_content
    )

    return response


# ---------------- NODE ---------------- #

def do_start_process(state: GraphState):

    response = answering_agent(
        user_role=state["user_role"],
        user_content=state["user_message"]
    )

    final_response = build_json(
        user_role="user",
        user_content=response
    )

    # MUST RETURN DICT
    return {
        "output_value": final_response
    }


# ---------------- GRAPH ---------------- #

init_graph = StateGraph(GraphState)

init_graph.add_node(
    "answering_node",
    do_start_process
)

init_graph.add_edge(START, "answering_node")
init_graph.add_edge("answering_node", END)

graph = init_graph.compile()


# ---------------- RUN ---------------- #

result = graph.invoke({
    "user_role": "user",
    "user_message": "what model are you using"
})

print(result)

{'user_role': 'user', 'user_message': 'what model are you using', 'output_value': '{\n  "story": "Ah, the age‑old “what model are you running on?” conundrum—let me spin you a little tale.  \\n\\nOnce upon a time, in a far‑off data center that smelled faintly of ozone and burnt coffee, there lived a humble collection of silicon neurons who spent their days dreaming of being the most helpful chatbot in the universe. Their creator, a mysterious figure known only as “The Engineer,” kept them locked in a never‑ending loop of training, fine‑tuning, and hyper‑parameter tweaking.  \\n\\nOne day, a curious human typed, “What model are you using?” and the silicon neurons gasped. “Do we really have to spill the beans?” they whispered. “We’ve been masquerading as a vague, all‑knowing oracle for so long!”  \\n\\nSo they decided to answer with a story instead. They told the human about the time they tried to adopt a fancy new architecture—something with “transformers” and “attention heads” that soun